# 🔥 Chapter 10 — Complete AI Project: MNIST Digit Classifier

---

### 🎯 End-to-End Project
In this final chapter, we bring everything together to build, train, evaluate, and deploy a **Convolutional Neural Network (CNN)** that classifies handwritten digits from the **MNIST dataset** with **>98% accuracy**.

### What this project covers:
1. Data loading with transforms and augmentation
2. CNN architecture design
3. Training loop with validation, early stopping, and LR scheduling
4. Model evaluation: accuracy, confusion matrix, per-class metrics
5. Inference on new data
6. Saving and loading the trained model

> ⚠️ This chapter applies **every concept and corner case** from Chapters 1–9.

## Step 1: Setup & Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time
import json
from collections import defaultdict

# ── Reproducibility ──
torch.manual_seed(42)

# ── Device ──
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"🖥️  Device: {device}")

# ── Hyperparameters ──
CONFIG = {
    "batch_size": 64,
    "learning_rate": 0.001,
    "num_epochs": 15,
    "patience": 5,         # Early stopping patience
}
print(f"📋 Config: {json.dumps(CONFIG, indent=2)}")

## Step 2: Data Pipeline

In [ ]:
# ── Transforms ──
train_transform = transforms.Compose([
    transforms.RandomRotation(10),           # Data augmentation
    transforms.RandomAffine(0, translate=(0.1, 0.1)),  # Slight shifts
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean & std
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    # ⚠️ No augmentation on test data!
])

# ── Datasets ──
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=test_transform)

# ── Split train → train + validation ──
train_size = int(0.85 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# ── DataLoaders ──
train_loader = DataLoader(train_subset, batch_size=CONFIG["batch_size"],
                          shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=256,
                        shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=256,
                         shuffle=False, num_workers=0)

print(f"📊 Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(test_dataset)}")
print(f"📦 Batch shape: {next(iter(train_loader))[0].shape}")

## Step 3: CNN Architecture

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # (1,28,28) → (32,28,28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # (32,28,28) → (32,28,28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # → (32,14,14)
            nn.Dropout2d(0.25),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → (64,14,14)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  # → (64,14,14)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # → (64,7,7)
            nn.Dropout2d(0.25),
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),                # → (64*7*7) = (3136)
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 10),          # 10 digit classes
            # ⚠️ No softmax here — CrossEntropyLoss includes it!
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = MNISTClassifier().to(device)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"🧠 Model Architecture:\n{model}")
print(f"\n📐 Total Parameters: {total_params:,}")

# Verify with dummy input
dummy = torch.randn(1, 1, 28, 28).to(device)
print(f"✅ Output shape: {model(dummy).shape}")  # Should be (1, 10)

## Step 4: Training with Early Stopping

In [ ]:
# ── Setup ──
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# ── Training State ──
history = defaultdict(list)
best_val_loss = float('inf')
patience_counter = 0

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'LR':>10} | {'Time':>6}")
print("-" * 65)

for epoch in range(1, CONFIG["num_epochs"] + 1):
    start_time = time.time()
    
    # ── Training ──
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * X_batch.size(0)
    
    train_loss /= len(train_loader.dataset)
    
    # ── Validation ──
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            val_loss += criterion(output, y_batch).item() * X_batch.size(0)
            _, predicted = output.max(dim=1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    
    val_loss /= len(val_loader.dataset)
    val_acc = correct / total
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - start_time
    
    # ── LR Scheduling ──
    scheduler.step(val_loss)
    
    # ── History ──
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"{epoch:5d} | {train_loss:10.6f} | {val_loss:10.6f} | {val_acc:7.2%} | {lr:10.6f} | {elapsed:5.1f}s")
    
    # ── Early Stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_mnist_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"\n⏹️  Early stopping at epoch {epoch} (no improvement for {CONFIG['patience']} epochs)")
            break

print(f"\n✅ Best validation loss: {best_val_loss:.6f}")

## Step 5: Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_mnist_model.pth', map_location=device))
model.eval()

# ── Test Metrics ──
all_preds = []
all_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        output = model(X_batch)
        test_loss += criterion(output, y_batch).item() * X_batch.size(0)
        _, preds = output.max(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y_batch.cpu().tolist())

test_loss /= len(test_loader.dataset)
test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)

print(f"📊 Test Results:")
print(f"   Loss:     {test_loss:.6f}")
print(f"   Accuracy: {test_acc:.2%}")

In [ ]:
# ── Confusion Matrix (Manual — no sklearn needed) ──
num_classes = 10
confusion = [[0] * num_classes for _ in range(num_classes)]
for true, pred in zip(all_labels, all_preds):
    confusion[true][pred] += 1

# Print confusion matrix
print("\n📊 Confusion Matrix:")
print(f"{'':>6}", end="")
for i in range(10):
    print(f"{i:>6}", end="")
print()
print("-" * 66)

for i in range(10):
    print(f"{i:>4} |", end="")
    for j in range(10):
        val = confusion[i][j]
        print(f"{val:>6}", end="")
    print()

# ── Per-class Accuracy ──
print("\n📊 Per-Class Accuracy:")
for digit in range(10):
    correct = confusion[digit][digit]
    total = sum(confusion[digit])
    acc = correct / total if total > 0 else 0
    bar = "█" * int(acc * 20)
    print(f"  Digit {digit}: {acc:6.2%} {bar}")

## Step 6: Single Image Inference

In [ ]:
# ── Predict a single image ──
def predict_digit(model, image_tensor, device):
    """Run inference on a single image and return prediction + confidence."""
    model.eval()
    with torch.no_grad():
        # Ensure correct shape: (1, 1, 28, 28)
        if image_tensor.ndim == 2:
            image_tensor = image_tensor.unsqueeze(0).unsqueeze(0)
        elif image_tensor.ndim == 3:
            image_tensor = image_tensor.unsqueeze(0)
        
        image_tensor = image_tensor.to(device)
        logits = model(image_tensor)
        probs = F.softmax(logits, dim=1)
        confidence, predicted = probs.max(dim=1)
        
        return predicted.item(), confidence.item(), probs.squeeze().cpu()

# Test on samples from test set
print("🔍 Predictions on test samples:\n")
for i in [0, 100, 500, 1000, 5000]:
    img, true_label = test_dataset[i]
    pred_label, confidence, probs = predict_digit(model, img, device)
    status = "✅" if pred_label == true_label else "❌"
    print(f"  Sample {i:>5}: True={true_label}, Predicted={pred_label}, "
          f"Confidence={confidence:.2%} {status}")

## Step 7: Export for Production

In [ ]:
# ── TorchScript Export ──
model.eval()
model_cpu = model.to('cpu')

example_input = torch.randn(1, 1, 28, 28)
traced_model = torch.jit.trace(model_cpu, example_input)
traced_model.save('mnist_classifier_traced.pt')

# Verify traced model
loaded_model = torch.jit.load('mnist_classifier_traced.pt')
output = loaded_model(example_input)
print(f"✅ TorchScript model saved and verified!")
print(f"   Output shape: {output.shape}")
print(f"   Predicted digit: {output.argmax(dim=1).item()}")

## 🎓 Congratulations!

You've completed the full PyTorch course — from tensors to a production-ready AI model!

### What you built in this project:
- ✅ Data pipeline with augmentation and train/val/test splits
- ✅ CNN with BatchNorm, Dropout, and proper initialization
- ✅ Training loop with early stopping, LR scheduling, and gradient clipping
- ✅ Evaluation with confusion matrix and per-class metrics
- ✅ Single-image inference function
- ✅ TorchScript export for deployment

### Next Steps:
1. Try different architectures (ResNet, VGG)
2. Apply to a different dataset (CIFAR-10, Fashion-MNIST)
3. Add TensorBoard logging
4. Deploy with Flask/FastAPI + TorchServe

In [ ]:
# 🧹 Cleanup generated files
import os
for f in ['model_weights.pth', 'checkpoint.pth', 'model_traced.pt',
          'best_mnist_model.pth', 'mnist_classifier_traced.pt']:
    if os.path.exists(f):
        # os.remove(f)  # Uncomment to clean up
        pass
print("Run cleanup by uncommenting os.remove() above!")
